# レッスン12 - エージェントスクラッチパッドによるチャット履歴の削減

このノートブックでは、Microsoft Agent Framework を使用して長い会話におけるコンテキスト管理を行う方法を示します。会話が増えるにつれてトークン数も増加し、最終的にモデルのコンテキストウィンドウを超えてしまいます。これには、<strong>コンテキスト要約パターン</strong>と永続的なメモリとなる<strong>エージェントスクラッチパッド</strong>で対応します。

## 学習内容:
1. <strong>なぜコンテキスト管理が重要か</strong>: トークン制限とコンテキストウィンドウの理解
2. <strong>コンテキスト対応エージェント</strong>: 自身の会話コンテキストを管理するエージェント構築
3. <strong>コンテキスト要約パターン</strong>: 会話履歴を圧縮するためのツール利用
4. <strong>エージェントスクラッチパッド</strong>: コンテキスト削減後も残る永続メモリ

## 前提条件:
- Azure OpenAI のセットアップと環境変数の設定が完了していること
- 前のレッスンでの基本的なエージェント概念の理解


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## コンテキスト管理が重要な理由

すべてのLLMには有限の<strong>コンテキストウィンドウ</strong>があります — これは1回のリクエストで処理できる最大トークン数です。複数ターンの会話が進行する中で：

- <strong>トークン数はユーザーメッセージとアシスタントの応答ごとに線形に増加</strong>します。
- <strong>プロンプトトークンがコストの大部分を占めます</strong>。なぜなら全履歴が毎ターン再送されるからです。
- 最終的に会話が<strong>コンテキストウィンドウを超過</strong>し、モデルは切り捨てるかエラーになります。

### コンテキスト管理の戦略

| 戦略 | 動作 | トレードオフ |
|---|---|---|
| <strong>切り捨て</strong> | 最も古いメッセージを削除 | 初期コンテキストを失う |
| <strong>要約</strong> | 古いメッセージをまとめて要約にする | 詳細は一部失われるが重要点は保持される |
| **スクラッチパッド / 外部メモリ** | 会話外に重要な事実を保存 | ツール呼び出しが必要だがどんな縮小にも耐えられる |

このノートブックでは<strong>要約</strong>と<strong>スクラッチパッドツール</strong>を組み合わせて、会話履歴が凝縮されてもエージェントが連続性を維持できるようにします。


## コンテキスト対応エージェントの作成


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## 長い会話のシミュレーション

複数回のやり取りがある会話を通して、どのようにコンテキストが蓄積されるか見てみましょう。エージェントは主要な詳細（好み、予算、旅行日程）を各ターンで保持し、継続性を示す必要があります。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

エージェントが以前のやり取りからコンテキストを保持していることに注目してください — 日本、寿司、寺院、写真、3000ドルの予算、一人旅、4月の期間について知っています。短い会話ではこれがうまく機能しますが、会話が長くなると全履歴を再送するのはコストがかかります。

より多くのやり取りで会話を続けて、コンテキストの蓄積を見てみましょう：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## コンテキスト要約パターン

会話が進むにつれて、<strong>要約ツール</strong>を使って蓄積されたコンテキストをコンパクトな形式にまとめることができます。エージェントはこのツールを呼び出して重要な好みを記録し、古いメッセージが削除されても本質的な情報が保持されるようにします。

このパターンは、より高度な履歴縮小の基礎となります:
1. エージェントが会話から重要な事実を特定します
2. それらを保持するために要約ツールを呼び出します
3. 要約が重要な情報を捉えているため、古いメッセージは安全に削除できます

以下に、エージェントが学習した内容のコンパクトな要約を記録するために呼び出せる `summarize_preferences` ツールを定義します。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## まとめ

このレッスンでは、Microsoft Agent Framework を使用して長時間のエージェント会話におけるコンテキスト管理方法を学びました:

### 重要なコンセプト
- <strong>コンテキストウィンドウは有限です</strong> — 会話履歴のすべてのトークンはコストがかかり、制限にカウントされます。
- <strong>要約ツール</strong> により、蓄積されたコンテキストをコンパクトな要約に凝縮し、トークン使用量を削減しつつ重要な情報を保持できます。
- <strong>エージェントのスクラッチパッド</strong> は、会話の削減に影響されない永続的な外部メモリを提供します。

### 作成したもの
- 複数ターンの会話にわたり連続性を維持する <strong>コンテキスト対応エージェント</strong>
- 重要なユーザー情報をコンパクトな形式で記録する <strong>要約ツール</strong> (`summarize_preferences`)
- コンテキストの保持と変更処理を示す <strong>複数ターンの会話</strong>

### 実際の応用例
- <strong>カスタマーサービスボット</strong>: 長時間のサポートセッションで利用者の好みを記憶
- <strong>パーソナルアシスタント</strong>: コンテキストを再説明することなく進行中のプロジェクトを追跡
- <strong>教育用チューター</strong>: 多数の対話を通じて生徒の進捗を維持

### 次のステップ
- ファイルベースの永続性を持つ完全なスクラッチパッドツールを実装する
- 要約後の自動履歴切り詰めを追加する
- セマンティックメモリ検索のためにベクターデータベースと組み合わせる
- 数日後でも完全なコンテキストで会話を再開できるエージェントを構築する


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
